In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import re
import json
from sklearn.model_selection import train_test_split
from pathlib import Path

In [ ]:
# ĐƯỜNG DẪN FILE GỐC TRÊN GOOGLE DRIVE
input_file = "/content/drive/MyDrive/NLP_FINAL/DATA/dataset_title_content_over800_length_fixed.xlsx"

# THƯ MỤC LƯU KẾT QUẢ
output_dir = Path("/content/drive/MyDrive/NLP_FINAL/sourcecode")
output_dir.mkdir(parents=True, exist_ok=True)

print("Input file:", input_file)
print("Output folder:", output_dir)

Input file: /content/drive/MyDrive/NLP_FINAL/DATA/dataset_title_content_over800_length_fixed.xlsx
Output folder: /content/drive/MyDrive/NLP_FINAL/sourcecode


In [ ]:
df = pd.read_excel(input_file)

print("Kích thước dữ liệu ban đầu:", df.shape)
print("Các cột hiện có:", list(df.columns))
df.head()

Kích thước dữ liệu ban đầu: (1602, 3)
Các cột hiện có: ['title', 'content', 'summary']


,title,content,summary
0,10 năm C.P. Củ Chi chuẩn hóa quy trình sản xuấ...,"Lễ kỷ niệm chủ đề ""Nhà máy C.P. Củ Chi: 10 năm...",Lễ kỷ niệm 10 năm Nhà máy C.P. Củ Chi diễn ra ...
1,10 triệu trái phiếu DNSE chính thức lên sàn HNX,10 triệu trái phiếu doanh nghiệp niêm yết có k...,DNSE phát hành thành công 10 triệu trái phiếu ...
2,11 điểm đổ xăng thay thế các trạm đóng cửa trê...,"Các điểm thay thế nằm dọc Xa lộ Hà Nội, Nguyễn...","Các điểm thay thế nằm dọc Xa lộ Hà Nội, Nguyễn..."
3,11 đơn vị xin cấp phép sản xuất vàng miếng,"Tại họp báo chiều 14/4, ông Đào Xuân Tuấn, Vụ ...","Tại họp báo chiều 14/4, ông Đào Xuân Tuấn, Vụ ..."
4,15 tiêu chí chọn cổ phiếu của Philip Fisher,Philip Fisher (1907-2004) là một trong những n...,Philip Fisher (1907-2004) là một trong những n...


In [ ]:
def clean_whitespace(text):
    if pd.isna(text):
        return ""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()

def normalize_text(text):
    text = clean_whitespace(text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    return text.strip()

def word_count(text):
    if pd.isna(text):
        return 0
    return len(str(text).split())

number_pattern = re.compile(
    r"""
    \b\d{1,2}/\d{1,2}/\d{2,4}\b
    |
    \b\d{1,3}(?:[.,]\d{3})*(?:[.,]\d+)?\s?(?:%|tỷ|triệu|nghìn|ngàn|USD|usd|đồng|cổ\sphiếu|ha|năm|tháng)?\b
    """,
    re.VERBOSE | re.IGNORECASE,
)

def extract_numbers(text):
    if pd.isna(text):
        return []
    return [normalize_text(x) for x in number_pattern.findall(str(text))]

In [ ]:
required_cols = ["title", "content", "summary"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Thiếu cột bắt buộc: {col}")

for col in required_cols:
    df[col] = df[col].fillna("").astype(str).map(normalize_text)

# Bỏ dòng rỗng
df = df[
    (df["title"] != "") &
    (df["content"] != "") &
    (df["summary"] != "")
].copy()

# Bỏ trùng
df = df.drop_duplicates(subset=["title", "content"]).reset_index(drop=True)

# Tạo cột thống kê
df["source_word_count"] = df["content"].map(word_count)
df["summary_word_count"] = df["summary"].map(word_count)
df["length_ratio"] = df["summary_word_count"] / df["source_word_count"]

# TÁCH RIÊNG SỐ TRONG TITLE, CONTENT, SUMMARY
df["numbers_in_title"] = df["title"].map(extract_numbers)
df["numbers_in_content"] = df["content"].map(extract_numbers)
df["numbers_in_summary"] = df["summary"].map(extract_numbers)

# GỘP TITLE + CONTENT ĐỂ KIỂM TRA SỐ LIỆU
df["numbers_in_source"] = df.apply(
    lambda row: list(set(row["numbers_in_title"] + row["numbers_in_content"])),
    axis=1
)

# So khớp số liệu: mọi số trong summary phải nằm trong title hoặc content
df["number_match"] = df.apply(
    lambda row: set(row["numbers_in_summary"]).issubset(set(row["numbers_in_source"])),
    axis=1
)

# Kiểm tra độ dài 20–25%
df["length_ok"] = df["length_ratio"].between(0.20, 0.25)

# Trạng thái duyệt
def get_review_status(row):
    if row["number_match"] and row["length_ok"]:
        return "approved"
    return "needs_revision"

df["review_status"] = df.apply(get_review_status, axis=1)

print("Kích thước sau làm sạch:", df.shape)
df.head()



Kích thước sau làm sạch: (1580, 13)


,title,content,summary,source_word_count,summary_word_count,length_ratio,numbers_in_title,numbers_in_content,numbers_in_summary,numbers_in_source,number_match,length_ok,review_status
0,10 năm C.P. Củ Chi chuẩn hóa quy trình sản xuấ...,"Lễ kỷ niệm chủ đề ""Nhà máy C.P. Củ Chi: 10 năm...",Lễ kỷ niệm 10 năm Nhà máy C.P. Củ Chi diễn ra ...,820,205,0.250000,[10 năm],"[10 năm, 26, 3, 10 năm, 10 năm, 63.000, 33.000...","[10 năm, 26, 3, 10 năm, 63.000, 33.000, 10, 15...","[63.000, 9.100, 5, 17, 26, 150, 10 năm, 1,9, 3...",True,True,approved
1,10 triệu trái phiếu DNSE chính thức lên sàn HNX,10 triệu trái phiếu doanh nghiệp niêm yết có k...,DNSE phát hành thành công 10 triệu trái phiếu ...,555,131,0.236036,[10 triệu],"[10 triệu, 24 tháng, 8,3, 3,5, 6 tháng, 10, 10...","[10 triệu, 24 tháng, 100.000 đồng, 8,3, 3,5, 6...","[2, 6 tháng, 11, 5, 3,5, 10 triệu, 21, 5 năm, ...",True,True,approved
2,11 điểm đổ xăng thay thế các trạm đóng cửa trê...,"Các điểm thay thế nằm dọc Xa lộ Hà Nội, Nguyễn...","Các điểm thay thế nằm dọc Xa lộ Hà Nội, Nguyễn...",357,83,0.232493,[11],"[51, 41, 100, 24, 24, 41, 100]","[51, 41, 100]","[11, 41, 100, 51, 24]",True,True,approved
3,11 đơn vị xin cấp phép sản xuất vàng miếng,"Tại họp báo chiều 14/4, ông Đào Xuân Tuấn, Vụ ...","Tại họp báo chiều 14/4, ông Đào Xuân Tuấn, Vụ ...",404,93,0.230198,[11],"[14, 4, 1.000 tỷ, 50.000 tỷ, 38, 1.000 tỷ, 50....","[14, 4, 1.000 tỷ, 50.000 tỷ, 38, 15]","[11, 50.000 tỷ, 38, 1.000 tỷ, 14, 4, 19,18 tri...",True,True,approved
4,15 tiêu chí chọn cổ phiếu của Philip Fisher,Philip Fisher (1907-2004) là một trong những n...,Philip Fisher (1907-2004) là một trong những n...,949,237,0.249737,[15],"[15, 4]",[15],"[4, 15]",True,True,approved


In [ ]:
output_file = "/content/drive/MyDrive/NLP_FINAL/DATA/dataset_checked_manual_review.xlsx"
df.to_excel(output_file, index=False)

print("Đã lưu file:", output_file)

Đã lưu file: /content/drive/MyDrive/NLP_FINAL/DATA/dataset_checked_manual_review.xlsx


In [ ]:
print("Tổng số mẫu:", len(df))
print("Số mẫu đúng độ dài:", int(df["length_ok"].sum()))
print("Số mẫu đúng số liệu:", int(df["number_match"].sum()))
print("Số mẫu approved:", int((df["review_status"] == "approved").sum()))

Tổng số mẫu: 1580
Số mẫu đúng độ dài: 1580
Số mẫu đúng số liệu: 1569
Số mẫu approved: 1569


In [ ]:
# LỌC DỮ LIỆU: CHỈ GIỮ CÁC DÒNG ĐƯỢC APPROVED
# Notebook phía trên đã tạo cột review_status:
# - "approved": đạt cả number_match và length_ok
# - "needs_revision": cần sửa lại
#
# Nếu file của bạn có cột trạng thái thủ công khác, đoạn này vẫn cố gắng nhận diện
# các giá trị như: approve, approved, duyệt, da duyet, ok.

status_col_candidates = [
    "review_status",
    "status",
    "approval_status",
    "label",
    "manual_review",
    "kiem_tra",
    "trang_thai"
]

status_col = next((c for c in status_col_candidates if c in df.columns), None)

if status_col is not None:
    approved_values = {
        "approve",
        "approved",
        "duyet",
        "duyệt",
        "da duyet",
        "đã duyệt",
        "ok",
        "pass",
        "passed"
    }

    df_clean = df[
        df[status_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(approved_values)
    ].copy().reset_index(drop=True)

    print(f"Đã lọc theo cột trạng thái: {status_col}")
else:
    # Trường hợp không có cột trạng thái, dùng đúng logic approved:
    # approved = đúng số liệu + đúng độ dài 20–25%
    if not {"number_match", "length_ok"}.issubset(df.columns):
        raise ValueError(
            "Không tìm thấy cột trạng thái duyệt, đồng thời cũng thiếu number_match/length_ok để tự lọc approved."
        )

    df_clean = df[
        (df["number_match"] == True) &
        (df["length_ok"] == True)
    ].copy().reset_index(drop=True)

    print("Không tìm thấy cột trạng thái thủ công, đã lọc theo number_match=True và length_ok=True.")

print("Số dòng ban đầu:", len(df))
print("Số dòng approved sau khi lọc:", len(df_clean))

if len(df_clean) == 0:
    raise ValueError("Không có dòng approved nào sau khi lọc. Hãy kiểm tra lại giá trị trong cột trạng thái.")

df_clean.head()


Đã lọc theo cột trạng thái: review_status
Số dòng ban đầu: 1580
Số dòng approved sau khi lọc: 1569


,title,content,summary,source_word_count,summary_word_count,length_ratio,numbers_in_title,numbers_in_content,numbers_in_summary,numbers_in_source,number_match,length_ok,review_status
0,10 năm C.P. Củ Chi chuẩn hóa quy trình sản xuấ...,"Lễ kỷ niệm chủ đề ""Nhà máy C.P. Củ Chi: 10 năm...",Lễ kỷ niệm 10 năm Nhà máy C.P. Củ Chi diễn ra ...,820,205,0.250000,[10 năm],"[10 năm, 26, 3, 10 năm, 10 năm, 63.000, 33.000...","[10 năm, 26, 3, 10 năm, 63.000, 33.000, 10, 15...","[63.000, 9.100, 5, 17, 26, 150, 10 năm, 1,9, 3...",True,True,approved
1,10 triệu trái phiếu DNSE chính thức lên sàn HNX,10 triệu trái phiếu doanh nghiệp niêm yết có k...,DNSE phát hành thành công 10 triệu trái phiếu ...,555,131,0.236036,[10 triệu],"[10 triệu, 24 tháng, 8,3, 3,5, 6 tháng, 10, 10...","[10 triệu, 24 tháng, 100.000 đồng, 8,3, 3,5, 6...","[2, 6 tháng, 11, 5, 3,5, 10 triệu, 21, 5 năm, ...",True,True,approved
2,11 điểm đổ xăng thay thế các trạm đóng cửa trê...,"Các điểm thay thế nằm dọc Xa lộ Hà Nội, Nguyễn...","Các điểm thay thế nằm dọc Xa lộ Hà Nội, Nguyễn...",357,83,0.232493,[11],"[51, 41, 100, 24, 24, 41, 100]","[51, 41, 100]","[11, 41, 100, 51, 24]",True,True,approved
3,11 đơn vị xin cấp phép sản xuất vàng miếng,"Tại họp báo chiều 14/4, ông Đào Xuân Tuấn, Vụ ...","Tại họp báo chiều 14/4, ông Đào Xuân Tuấn, Vụ ...",404,93,0.230198,[11],"[14, 4, 1.000 tỷ, 50.000 tỷ, 38, 1.000 tỷ, 50....","[14, 4, 1.000 tỷ, 50.000 tỷ, 38, 15]","[11, 50.000 tỷ, 38, 1.000 tỷ, 14, 4, 19,18 tri...",True,True,approved
4,15 tiêu chí chọn cổ phiếu của Philip Fisher,Philip Fisher (1907-2004) là một trong những n...,Philip Fisher (1907-2004) là một trong những n...,949,237,0.249737,[15],"[15, 4]",[15],"[4, 15]",True,True,approved


In [ ]:
output_xlsx = "/content/drive/MyDrive/NLP_FINAL/DATA/dataset_clean.xlsx"
output_csv = "/content/drive/MyDrive/NLP_FINAL/DATA/dataset_clean.csv"

df_clean.to_excel(output_xlsx, index=False)
df_clean.to_csv(output_csv, index=False, encoding="utf-8-sig")

print("Đã lưu:")
print(output_xlsx)
print(output_csv)

Đã lưu:
/content/drive/MyDrive/NLP_FINAL/DATA/dataset_clean.xlsx
/content/drive/MyDrive/NLP_FINAL/DATA/dataset_clean.csv


In [ ]:
keep_cols = [
    "title",
    "content",
    "summary",
    "source_word_count",
    "summary_word_count",
    "length_ratio",
    "number_match"
]

keep_cols = [c for c in keep_cols if c in df_clean.columns]
df_model = df_clean[keep_cols].copy().reset_index(drop=True)

print(df_model.shape)
df_model.head()

(1569, 7)


,title,content,summary,source_word_count,summary_word_count,length_ratio,number_match
0,10 năm C.P. Củ Chi chuẩn hóa quy trình sản xuấ...,"Lễ kỷ niệm chủ đề ""Nhà máy C.P. Củ Chi: 10 năm...",Lễ kỷ niệm 10 năm Nhà máy C.P. Củ Chi diễn ra ...,820,205,0.250000,True
1,10 triệu trái phiếu DNSE chính thức lên sàn HNX,10 triệu trái phiếu doanh nghiệp niêm yết có k...,DNSE phát hành thành công 10 triệu trái phiếu ...,555,131,0.236036,True
2,11 điểm đổ xăng thay thế các trạm đóng cửa trê...,"Các điểm thay thế nằm dọc Xa lộ Hà Nội, Nguyễn...","Các điểm thay thế nằm dọc Xa lộ Hà Nội, Nguyễn...",357,83,0.232493,True
3,11 đơn vị xin cấp phép sản xuất vàng miếng,"Tại họp báo chiều 14/4, ông Đào Xuân Tuấn, Vụ ...","Tại họp báo chiều 14/4, ông Đào Xuân Tuấn, Vụ ...",404,93,0.230198,True
4,15 tiêu chí chọn cổ phiếu của Philip Fisher,Philip Fisher (1907-2004) là một trong những n...,Philip Fisher (1907-2004) là một trong những n...,949,237,0.249737,True


In [ ]:
import re

def clean_whitespace(text):
    if pd.isna(text):
        return ""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()

def normalize_text(text):
    text = clean_whitespace(text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    return text.strip()

for col in ["title", "content", "summary"]:
    df_model[col] = df_model[col].fillna("").astype(str).map(normalize_text)

print("Số dòng ban đầu:", len(df_model))

df_clean = df_model[
    (df_model["title"] != "") &
    (df_model["content"] != "") &
    (df_model["summary"] != "")
].copy()

print("Sau khi xóa dòng rỗng:", len(df_clean))

df_clean = df_clean.drop_duplicates(subset=["title"], keep="first").reset_index(drop=True)

print("Sau khi xóa trùng title:", len(df_clean))
df_clean.head()

Số dòng ban đầu: 1569
Sau khi xóa dòng rỗng: 1569
Sau khi xóa trùng title: 1556


,title,content,summary,source_word_count,summary_word_count,length_ratio,number_match
0,10 năm C.P. Củ Chi chuẩn hóa quy trình sản xuấ...,"Lễ kỷ niệm chủ đề ""Nhà máy C.P. Củ Chi: 10 năm...",Lễ kỷ niệm 10 năm Nhà máy C.P. Củ Chi diễn ra ...,820,205,0.250000,True
1,10 triệu trái phiếu DNSE chính thức lên sàn HNX,10 triệu trái phiếu doanh nghiệp niêm yết có k...,DNSE phát hành thành công 10 triệu trái phiếu ...,555,131,0.236036,True
2,11 điểm đổ xăng thay thế các trạm đóng cửa trê...,"Các điểm thay thế nằm dọc Xa lộ Hà Nội, Nguyễn...","Các điểm thay thế nằm dọc Xa lộ Hà Nội, Nguyễn...",357,83,0.232493,True
3,11 đơn vị xin cấp phép sản xuất vàng miếng,"Tại họp báo chiều 14/4, ông Đào Xuân Tuấn, Vụ ...","Tại họp báo chiều 14/4, ông Đào Xuân Tuấn, Vụ ...",404,93,0.230198,True
4,15 tiêu chí chọn cổ phiếu của Philip Fisher,Philip Fisher (1907-2004) là một trong những n...,Philip Fisher (1907-2004) là một trong những n...,949,237,0.249737,True


In [ ]:
# CHIA TRAIN / VALIDATION / TEST THEO SỐ LƯỢNG MẪU THỰC TẾ
#
# Ưu tiên:
# 1. Nếu đủ >= 1000 mẫu approved: chia đúng tỉ lệ 80/10/10.
# 2. Nếu chưa đủ 1000 mẫu nhưng vẫn đủ yêu cầu tối thiểu train >= 500 và test >= 100:
#    giữ test = 100, train >= 500, phần còn lại làm validation.
# 3. Nếu không đủ train >= 500 và test >= 100: báo lỗi để bạn bổ sung dữ liệu.

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
MIN_TRAIN = 500
MIN_TEST = 100

df_clean = df_clean.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
n = len(df_clean)

print("Tổng số mẫu approved dùng để chia:", n)

if n >= 1000:
    # Chia đúng 80/10/10 và đảm bảo test khoảng >=100 mẫu
    train_df, temp_df = train_test_split(
        df_clean,
        test_size=0.20,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    split_strategy = "Strict 80/10/10"

else:
    # Trường hợp số mẫu chưa đủ để 10% test đạt >=100.
    # Khi đó ưu tiên đáp ứng yêu cầu tối thiểu của đề: train >=500, test >=100.
    if n < MIN_TRAIN + MIN_TEST + 1:
        raise ValueError(
            f"Dữ liệu approved hiện có {n} mẫu, chưa đủ để chia với train >= {MIN_TRAIN}, "
            f"test >= {MIN_TEST} và vẫn có validation. Cần ít nhất {MIN_TRAIN + MIN_TEST + 1} mẫu."
        )

    test_size = MIN_TEST
    val_size = max(1, round(n * 0.10))

    # Nếu val quá lớn làm train < 500 thì giảm val xuống phần còn lại
    if n - test_size - val_size < MIN_TRAIN:
        val_size = max(1, n - test_size - MIN_TRAIN)

    train_size = n - val_size - test_size

    if train_size < MIN_TRAIN:
        raise ValueError(
            f"Sau khi chia, train chỉ có {train_size} mẫu. Hãy bổ sung dữ liệu approved."
        )

    train_df = df_clean.iloc[:train_size].copy()
    val_df = df_clean.iloc[train_size:train_size + val_size].copy()
    test_df = df_clean.iloc[train_size + val_size:].copy()

    split_strategy = "Minimum requirement split: train >=500, test=100, val=phần còn lại"

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

df_final = pd.concat([train_df, val_df, test_df], ignore_index=True)

print("Chiến lược chia:", split_strategy)
print(df_final["split"].value_counts())
print("\nTỉ lệ thực tế:")
print((df_final["split"].value_counts(normalize=True) * 100).round(2).astype(str) + "%")

df_final.head()


Tổng số mẫu approved dùng để chia: 1556
Chiến lược chia: Strict 80/10/10
split
train    1244
val       156
test      156
Name: count, dtype: int64

Tỉ lệ thực tế:
split
train    79.95%
val      10.03%
test     10.03%
Name: proportion, dtype: object


,title,content,summary,source_word_count,summary_word_count,length_ratio,number_match,split
0,Các tỷ phú Việt có thêm 2 tỷ USD trong phiên c...,"VN-Index, chỉ số đại diện cho sàn TP HCM, hôm ...","VN-Index, chỉ số đại diện cho sàn TP HCM, hôm ...",589,146,0.247878,True,train
1,Quảng Ninh hỗ trợ hai triệu đồng mỗi tháng cho...,"Riêng người đang ở tại đảo Quan Lạn, Minh Châu...",Hiện tỉnh Quảng Ninh có hơn 1.000 người nằm tr...,202,49,0.242574,True,train
2,BIDV huy động thành công khoản vay bền vững 25...,Đây là khoản vay hợp vốn với sự tham gia của n...,Đây là khoản vay hợp vốn với sự tham gia của n...,648,154,0.237654,True,train
3,TP HCM chuẩn bị mở đường 6 làn xe ở cửa ngõ tâ...,Dự án vừa được UBND TP HCM giao Sở Tài chính p...,"Công trình dự kiến khởi công cuối năm 2026, ho...",206,44,0.213592,True,train
4,Cổ phiếu 'lên sàn' sẽ nhanh hơn 3-6 lần trước đây,"Tại họp báo thường kỳ Chính phủ cuối tuần này,...",Nghị định 245/2025 được kỳ vọng rút ngắn đáng ...,660,164,0.248485,True,train


In [ ]:
# LƯU CÁC FILE TRAIN / VAL / TEST CHO MODEL
#
# Gộp title + content thành original_text để model nhìn thấy cả tiêu đề và nội dung.
# Việc này quan trọng nếu title có số liệu.

output_dir = "/content/drive/MyDrive/NLP_FINAL/DATA"

def build_original_text(row):
    title = str(row.get("title", "")).strip()
    content = str(row.get("content", "")).strip()

    # Xử lý trường hợp title/content bị NaN
    if title.lower() == "nan":
        title = ""
    if content.lower() == "nan":
        content = ""

    if title:
        return f"Tiêu đề: {title}\nNội dung: {content}"
    else:
        return content


# Tạo bản copy để không ảnh hưởng dữ liệu gốc
train_for_model = train_df.copy()
val_for_model = val_df.copy()
test_for_model = test_df.copy()

# Gộp title + content vào original_text
train_for_model["original_text"] = train_for_model.apply(build_original_text, axis=1)
val_for_model["original_text"] = val_for_model.apply(build_original_text, axis=1)
test_for_model["original_text"] = test_for_model.apply(build_original_text, axis=1)

# Chỉ giữ các cột cần cho model
# Giữ thêm title để tiện kiểm tra/phân tích lỗi sau này
train_for_model = train_for_model[["title", "original_text", "summary"]].copy()
val_for_model = val_for_model[["title", "original_text", "summary"]].copy()
test_for_model = test_for_model[["title", "original_text", "summary"]].copy()

# Lưu file cho model
train_for_model.to_csv(f"{output_dir}/train_for_model.csv", index=False, encoding="utf-8-sig")
val_for_model.to_csv(f"{output_dir}/val_for_model.csv", index=False, encoding="utf-8-sig")
test_for_model.to_csv(f"{output_dir}/test_for_model.csv", index=False, encoding="utf-8-sig")

# Lưu thêm file đầy đủ có cột split để phục vụ báo cáo/thống kê
df_final.to_csv(f"{output_dir}/dataset_final_with_split.csv", index=False, encoding="utf-8-sig")
df_final.to_excel(f"{output_dir}/dataset_final_with_split.xlsx", index=False)

print("Đã lưu các file:")
print(f"{output_dir}/train_for_model.csv:", train_for_model.shape)
print(f"{output_dir}/val_for_model.csv:", val_for_model.shape)
print(f"{output_dir}/test_for_model.csv:", test_for_model.shape)
print(f"{output_dir}/dataset_final_with_split.csv:", df_final.shape)

print("\nVí dụ original_text sau khi gộp:")
print(train_for_model["original_text"].iloc[0][:1000])

Đã lưu các file:
/content/drive/MyDrive/NLP_FINAL/DATA/train_for_model.csv: (1244, 3)
/content/drive/MyDrive/NLP_FINAL/DATA/val_for_model.csv: (156, 3)
/content/drive/MyDrive/NLP_FINAL/DATA/test_for_model.csv: (156, 3)
/content/drive/MyDrive/NLP_FINAL/DATA/dataset_final_with_split.csv: (1556, 8)

Ví dụ original_text sau khi gộp:
Tiêu đề: Các tỷ phú Việt có thêm 2 tỷ USD trong phiên chứng khoán tăng kỷ lục
Nội dung: VN-Index, chỉ số đại diện cho sàn TP HCM, hôm nay đóng cửa với mức tăng 79 điểm, đánh dấu phiên khởi sắc nhất lịch sử chứng khoán Việt Nam tính theo điểm số tuyệt đối. Nhờ đó, tài sản của các tỷ phú cũng tăng đáng kể.
Bảng xếp hạng tỷ phú theo thời gian thực của Forbes cho biết ông Phạm Nhật Vượng, Chủ tịch Vingroup, là người kiếm tiền nhiều tiền nhất Việt Nam trong ngày 8/4. Tài sản của doanh nhân này đã tăng 1,3 tỷ USD hôm nay, lên 25,7 tỷ USD. Ông hiện vẫn là người giàu nhất Đông Nam Á và xếp thứ 94 thế giới, tăng 6 bậc trong một ngày.
Xếp sau về số tiền kiếm được trong p